# Прототип инструмента получения информации по рейсам

Функция-прототип, описывающая работу одного из основных инструментов агента - получение информации по рейсам на определенную дату конкретного аэропорта через его IATA код. Для этого используется [API "Яндекс Расписаний"](https://yandex.ru/dev/rasp/doc/ru/reference/schedule-on-station)

## Импорт

In [1]:
import os
from dotenv import load_dotenv
from datetime import datetime

import requests

## Функция

In [2]:
def get_station_timetable_test(station_code: str,
                               event_date: str,
                               transport_types: str,
                               event: str) -> str:
    def get_hh_mm(dt_string):
        try:
            dt = datetime.fromisoformat(dt_string)
            return dt.strftime('%H:%M')
        except Exception as e:
            raise e
        
    url = "https://api.rasp.yandex-net.ru/v3.0/schedule/"
    params = {
        'apikey': os.getenv('YANDEX_API_KEY'),
        'station': station_code,
        'date': event_date,
        'transport_types': transport_types,
        'event': event,
        'system': 'iata',
    }
    try:
        response = requests.get(url, params=params)
    except:
        return f'Ошибка получения информации по API'
    data = response.json()
    
    date = data['date'] # Дата
    total_values = data['pagination']['total'] # Всего подходящих значений
    station_type = data['station']['station_type'] # Тип станции официальный на английском
    station_type_name = data['station']['station_type_name'] # Тип станции общепринятый на русском
    
    schedules = data['schedule']
    for schedule in schedules:
        even_time_org = schedule[event]
        event_time = get_hh_mm(even_time_org) # Время события
        trip_number = schedule['thread']['number'] # Номер рейса
        # trip_title = schedule['thread']['title'] # Название нитки. Составляется из полных названий первой и последней станций следования.
        departure_point, arrival_point = schedule['thread']['title'].split(' — ')
        transport_type = schedule['thread']['transport_type'] # Тип транспорта
        transport_name = schedule['thread']['vehicle'] # Название транспорта
        company_name = schedule['thread']['carrier']['title'] # Название компании-перевозчика
        
    print(f"""
          Дата события: {date}
          Всего значений: {total_values}
          Тип станций на английском: {station_type}
          Тип станций на русском: {station_type_name}
          
          Оригианльный формат времени: {even_time_org}
          Время события: {event_time}
          Номер рейса: {trip_number}
          Точка отправления: {departure_point}
          Точка прибытия: {arrival_point}
          Тип транспорта: {transport_type}
          Название тренспорта: {transport_name}
          Название компании: {company_name}
          """)

## Тестирование

In [3]:
get_station_timetable_test(station_code='OVB',
                           event_date='2026-06-11',
                           transport_types='plane',
                           event='arrival')


          Дата события: 2026-06-11
          Всего значений: 121
          Тип станций на английском: airport
          Тип станций на русском: аэропорт

          Оригианльный формат времени: 2026-06-11T20:30:00+07:00
          Время события: 20:30
          Номер рейса: S7 5046
          Точка отправления: Уфа
          Точка прибытия: Новосибирск
          Тип транспорта: plane
          Название тренспорта: Boeing 737-800
          Название компании: S7 Airlines
          


In [4]:
get_station_timetable_test(station_code='LAX',
                           event_date='2026-06-11',
                           transport_types='plane',
                           event='arrival')


          Дата события: 2026-06-11
          Всего значений: 764
          Тип станций на английском: airport
          Тип станций на русском: аэропорт

          Оригианльный формат времени: 2026-06-11T08:35:00-07:00
          Время события: 08:35
          Номер рейса: WN 1840
          Точка отправления: Юджин
          Точка прибытия: Лос-Анджелес
          Тип транспорта: plane
          Название тренспорта: None
          Название компании: Southwest Airlines
          


In [5]:
get_station_timetable_test(station_code='LAX',
                           event_date='2026-06-11',
                           transport_types='plane',
                           event='departure')


          Дата события: 2026-06-11
          Всего значений: 763
          Тип станций на английском: airport
          Тип станций на русском: аэропорт

          Оригианльный формат времени: 2026-06-11T07:35:00-07:00
          Время события: 07:35
          Номер рейса: WN 2395
          Точка отправления: Лос-Анджелес
          Точка прибытия: Эль-Пасо
          Тип транспорта: plane
          Название тренспорта: Boeing 737-800
          Название компании: Southwest Airlines
          
